# ChibiCreate — FLUX → Qwen

**PRIMARY EXPERIMENT:** `FLUX run_003` → Qwen, com `full_body` + `outfit` como
referências de design.

Segundo estágio: **Qwen-Image-Edit-2511** aproximando do chibi alvo uma saída
do FLUX que já preserva bem o design. A pergunta é se o Qwen puxa as
proporções para o chibi **sem** reinterpretar roupa, capa, cabelo, chifres e
ornamentos.

**O FLUX não roda aqui.** Você faz upload do ZIP com os resultados que já
executou.

### Por que `run_003` e não `run_001`

Decisão do usuário, por inspeção visual dos resultados FLUX:

| | chibi | design preservado |
|---|---|---|
| `run_001` | mais forte | reinterpretou mais roupa e design |
| **`run_003`** | menos chibi | **preserva estrutura, roupa, capa, cabelo, ornamentos** |

O estágio Qwen existe justamente para cobrir o que falta no `run_003`:
partir de um design fiel e puxar as proporções. Partir do `run_001` seria
pedir ao Qwen que reconstruísse um design já perdido no estágio 1.

---

## Experimento principal e comparações

| | Pipeline | image1 (editada) | image2 | image3 |
|---|---|---|---|---|
| **PRIMARY** | FLUX **run_003** → Qwen | **saída run_003** | `full_body` | `outfit` |
| A | Original → Qwen | `full_body` | — | — |
| B | FLUX run_003 → Qwen | saída run_003 | `full_body` | — |
| C | FLUX **run_001** → Qwen | saída run_001 | `full_body` | `outfit` |

A/B/C ficam como **comparação**: A isola o Qwen sozinho, B mede o efeito da
segunda referência, C mede o efeito de trocar a entrada do estágio 1. O
`face.png` não entra — o node Core só comporta três imagens no total, e não
há workaround sem custom node.

---

## ⚠️ Leia antes de escolher a GPU

O Qwen-Image-Edit-2511 é **muito** maior que o FLUX klein:

| | FLUX.2 klein 4B | Qwen-Image-Edit-2511 |
|---|---|---|
| Pesos (diffusion) | 7.75 GB | **20.5 GB** (fp8mixed) |
| Download total | ~16 GB | **~30 GB** |
| `min_vram_gb` do projeto | 13 | **24** |

**T4 (15 GB) não serve.** A célula de preflight bloqueia antes de gastar GPU.
Use **A100 (40 GB)**, ou L4 (22.5 GB) sabendo que fica abaixo do limiar de 24.

Não há fallback automático: nada de trocar de modelo, quantização ou
checkpoint por conta própria. Se não couber, o notebook **para**.

---

**Não é Flow 02. Não produz `master.png`. Nenhum artefato é aprovado.**


## 1 — GPU real

Registra o que a sessão de fato deu. Não assume nada.

In [ ]:
import json, subprocess, sys

out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'], capture_output=True, text=True)

if out.returncode != 0:
    raise SystemExit('SEM GPU. Ambiente de execucao > Alterar tipo > GPU.')

name, mem, driver = [x.strip() for x in out.stdout.strip().split(',')]
vram_gb = float(mem.split()[0]) / 1024

gpu_info = {'name': name, 'vram_mb': int(mem.split()[0]),
            'vram_gb': round(vram_gb, 1), 'driver': driver}
json.dump(gpu_info, open('/content/gpu_info.json', 'w'), indent=2)

print(f'GPU        : {name}')
print(f'VRAM       : {vram_gb:.1f} GB')
print(f'driver     : {driver}')
print(f'python     : {sys.version.split()[0]}')

MIN_VRAM_GB = 24  # vem de config/environments/cloud.yaml
if vram_gb + 0.5 < MIN_VRAM_GB:
    print()
    print('=' * 64)
    print(f'ATENCAO: {vram_gb:.1f} GB < {MIN_VRAM_GB} GB exigidos pelo projeto.')
    print('O Qwen fp8mixed sozinho ocupa ~20.5 GB de pesos.')
    print('NAO vou trocar de modelo nem de quantizacao automaticamente.')
    print('Troque para A100 (40 GB) ou aceite o risco de OOM em L4.')
    print('=' * 64)
else:
    print('\nVRAM suficiente para o alvo do projeto.')

## 2 — Clonar o ChibiCreate

Branch `arena/01a07ece-chibicreate` (a `main` não tem a pipeline).

In [ ]:
%cd /content
!git clone --branch arena/01a07ece-chibicreate \
    https://github.com/BloomRX/ChibiCreate.git 2>/dev/null || echo 'ja clonado'
%cd /content/ChibiCreate
!git log --oneline -1

import pathlib
for p in ['workflows/experimental/qwen_edit_multiref/v1.json',
          'characters/waifu_001/reference/full_body.png',
          'characters/waifu_001/reference/outfit.png']:
    print(('OK   ' if pathlib.Path(p).is_file() else 'FALTA'), p)

## 3 — Upload do ZIP com os resultados do FLUX

Envie o ZIP que você baixou do notebook do FLUX. Ele deve conter
`model_eval/flux2_klein_4b/run_003/output.png` — a entrada do
**experimento principal**.

O `run_001` também é localizado, mas apenas para o teste C de comparação. Se
faltar, o principal roda mesmo assim e o C é pulado.

A célula procura **runs nomeados** — nunca substitui por outro run em silêncio.


In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

dest = pathlib.Path('/content/flux_input')
if dest.exists():
    shutil.rmtree(dest)
dest.mkdir(parents=True)

print('Selecione o ZIP com os resultados do FLUX...')
up = files.upload()
zip_name = list(up)[0]

with zipfile.ZipFile(zip_name) as z:
    z.extractall(dest)

print('\nsaidas do FLUX encontradas no ZIP:')
for p in sorted(dest.rglob('run_*/output.png')):
    print('  ', p.relative_to(dest))


def achar(run):
    hits = sorted(dest.rglob(f'{run}/output.png'))
    if len(hits) > 1:
        raise SystemExit(
            f'PARE: {len(hits)} candidatos a {run}/output.png. Ambiguo — '
            'nao vou escolher por voce.')
    return hits[0].resolve() if hits else None


# Entrada do experimento PRINCIPAL. Sem ela nao ha o que rodar.
FLUX_RUN003 = achar('run_003')
if FLUX_RUN003 is None:
    raise SystemExit(
        'PARE: run_003/output.png nao encontrado no ZIP. Ele e a entrada do '
        'PRIMARY EXPERIMENT e nao pode ser substituido por outro run.')

# Apenas para o teste C de comparacao. Opcional.
FLUX_RUN001 = achar('run_001')

print('\nPRIMARY  (estagio 1):', FLUX_RUN003)
print('run_001 (so teste C) :', FLUX_RUN001 or 'ausente — teste C sera pulado')


### 3.1 — Verificar a saída do FLUX

Sem redimensionar, recomprimir ou editar: a imagem entra no Qwen exatamente
como saiu do FLUX. Registramos SHA256 do arquivo **e** pixel SHA256 (hashes
distintos: metadados PNG mudam o primeiro sem mudar um pixel).


In [ ]:
import hashlib, json
from PIL import Image
from IPython.display import display


def registrar(path, papel):
    data = path.read_bytes()
    with Image.open(path) as im:
        size, mode = im.size, im.mode
        pixel_sha = hashlib.sha256(im.convert('RGBA').tobytes()).hexdigest()
    info = {
        'file': str(path),
        'role': papel,
        'origin': f'upload do usuario: {zip_name} -> {path.parent.name}/output.png',
        'artifact_sha256': hashlib.sha256(data).hexdigest(),
        'pixel_sha256': pixel_sha,
        'width': size[0], 'height': size[1], 'mode': mode,
        'bytes': len(data),
        'modified_before_stage2': False,
    }
    print('=' * 68)
    for k, v in info.items():
        print(f'  {k:24} {v}')
    display(Image.open(path))
    return info


flux_baseline = {'primary': registrar(FLUX_RUN003, 'stage1_primary_run_003')}
if FLUX_RUN001 is not None:
    flux_baseline['run_001'] = registrar(FLUX_RUN001, 'stage1_alt_run_001')

json.dump(flux_baseline, open('/content/flux_baseline.json', 'w'), indent=2)


## 4 — Baixar o Qwen-Image-Edit-2511

~30 GB. É a etapa longa. Os hashes vêm de `config/models.lock.yaml`
(Comfy-Org, redistribuição oficial — não é quantização comunitária).

In [ ]:
!pip install -q huggingface_hub
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo 'ja clonado'
%cd /content/ComfyUI
!git rev-parse HEAD > /content/comfy_commit.txt
COMFY_COMMIT = open('/content/comfy_commit.txt').read().strip()
print('ComfyUI commit:', COMFY_COMMIT)
!pip install -q -r requirements.txt

from huggingface_hub import hf_hub_download
import shutil, pathlib

ALVOS = [
    ('Comfy-Org/Qwen-Image-Edit_ComfyUI',
     'split_files/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors',
     'models/diffusion_models',
     'c9fdc158e46d3b61ef75f21ae866ca2fe808bf4a53643120d1c1e87c19280a4e'),
    ('Comfy-Org/Qwen-Image_ComfyUI',
     'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors',
     'models/text_encoders',
     'cb5636d852a0ea6a9075ab1bef496c0db7aef13c02350571e388aea959c5c0b4'),
    ('Comfy-Org/Qwen-Image_ComfyUI',
     'split_files/vae/qwen_image_vae.safetensors',
     'models/vae',
     'a70580f0213e67967ee9c95f05bb400e8fb08307e017a924bf3441223e023d1f'),
]

registro = []
for repo, path, destdir, sha_esperado in ALVOS:
    print(f'\nbaixando {path.split("/")[-1]} ...')
    p = hf_hub_download(repo_id=repo, filename=path)
    d = pathlib.Path('/content/ComfyUI') / destdir
    d.mkdir(parents=True, exist_ok=True)
    final = d / pathlib.Path(path).name
    if not final.exists():
        shutil.copy(p, final)
    registro.append({'repo': repo, 'file': final.name,
                     'size_bytes': final.stat().st_size,
                     'sha256_expected': sha_esperado})
    print('  ->', final, f'{final.stat().st_size/1e9:.2f} GB')

import json
json.dump(registro, open('/content/model_record.json', 'w'), indent=2)

## 5 — Subir o ComfyUI

In [ ]:
import subprocess, time, urllib.request, json

%cd /content/ComfyUI
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=open('/content/comfy.log', 'w'), stderr=subprocess.STDOUT)

for i in range(180):
    try:
        r = urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3)
        stats = json.loads(r.read())
        print('ComfyUI no ar apos', i, 's')
        print(json.dumps(stats.get('system', {}), indent=2)[:400])
        break
    except Exception:
        time.sleep(1)
else:
    print(open('/content/comfy.log').read()[-3000:])
    raise SystemExit('ComfyUI nao subiu. Log acima.')

## 6 — Preflight real

Confere GPU, VRAM, modelos, workflow e licença **antes** de gastar GPU.

In [ ]:
import os
os.environ['CHIBI_COMFY_URL'] = 'http://127.0.0.1:8188'
%cd /content/ChibiCreate

!python -m scripts.chibi.cli comfy status --env cloud
print('=' * 62)
!python -m scripts.chibi.cli comfy preflight --env cloud
print('=' * 62)
!python -m scripts.chibi.cli comfy validate --env cloud \
    --workflow experimental/qwen_edit_multiref

### 6.1 — Topologia: as referências chegam ao conditioning?

Este é o modo de falhar mais perigoso do experimento: uma referência
desconectada não gera erro — a imagem sai plausível, só que a referência
nunca entrou. Aborta antes da GPU.

In [ ]:
import json

wf = json.load(open('workflows/experimental/qwen_edit_multiref/v1.json'))
n = {k: v for k, v in wf.items() if not k.startswith('_')}

enc = next(v for v in n.values()
           if v['class_type'] == 'TextEncodeQwenImageEditPlus'
           and 'image1' in v['inputs'])

assert enc['inputs']['image1'] == ['4', 0],  'image1 nao e a imagem principal'
assert enc['inputs']['image2'] == ['11', 0], 'image2 mal conectada'
assert enc['inputs']['image3'] == ['12', 0], 'image3 mal conectada'

ks = next(v for v in n.values() if v['class_type'] == 'KSampler')
lat = ks['inputs']['latent_image'][0]
assert n[lat]['class_type'] == 'VAEEncode', 'latente nao vem de VAEEncode'
assert n[lat]['inputs']['pixels'] == ['4', 0], 'latente nao vem da imagem principal'

print('OK  image1 = imagem a editar (e alimenta o latente inicial)')
print('OK  image2 / image3 = referencias de design')
print('OK  img2img real, nao canvas vazio')
print('\nLIMITE: o node Core aceita 3 imagens. Com a saida do FLUX em image1')
print('        sobram 2 slots — por isso o teste C usa full_body + outfit,')
print('        sem face.png. Passar 3 refs ABORTA (sem fallback).')

## 7 — Parâmetros da rodada

`DENOISE` é **hipótese de baseline**, não valor calibrado. Calibrar é uma
etapa separada, depois de olhar estes resultados.


In [ ]:
PROMPT_QWEN = (
    "Preserve the exact same character identity, face, hair, eyes, horns, body proportions, color palette, and overall chibi style from the input image. Preserve the original character design shown in the reference images, especially the clothing, cape, golden ornaments, accessories, and their shapes and placement. Do not redesign, replace, modernize, sexualize, simplify away, or invent clothing elements. Redraw the character as a clean polished game/gacha chibi character while keeping the original outfit design recognizable and faithful to the references. Only make changes necessary to adapt the original design naturally to the chibi proportions. Full body, front-facing, clean silhouette, consistent lineart, polished game art, soft shading, highly readable at small size. The output should look like the same original character converted into chibi form, not a new character inspired by the original."
)

# BASELINE_HYPOTHESIS — nao e valor validado nem otimizado.
# Nao ha denoise recomendado para Qwen-Edit registrado no projeto
# (o unico denoise existente e 1.0, do FLUX, para geracao a partir do zero).
# Aqui o objetivo e o OPOSTO: preservar a base e corrigir o design.
# 1.0 descartaria o chibi do FLUX e viraria uma geracao nova, anulando o
# experimento. 0.5 e o ponto medio conservador: espaco para corrigir roupa
# sem dissolver a personagem.
# A calibracao e uma etapa SEPARADA, depois de olhar estes resultados.
DENOISE = 0.5
DENOISE_STATUS = 'BASELINE_HYPOTHESIS'

SEED = 42

print('prompt   :', len(PROMPT_QWEN), 'caracteres')
print('denoise  :', DENOISE, f'({DENOISE_STATUS})')
print('seed     :', SEED)
print('negativo : nenhum (por instrucao — medir o comportamento base)')

In [ ]:
# Painel de confirmacao — o que vai rodar no PRIMARY EXPERIMENT.
print('=' * 68)
print('PRIMARY EXPERIMENT')
print('=' * 68)
print('SOURCE:')
print('    run_003/output.png')
print(f'    {FLUX_RUN003}')
print(f'    pixel_sha256 {flux_baseline["primary"]["pixel_sha256"][:32]}...')
print(f'    {flux_baseline["primary"]["width"]}x{flux_baseline["primary"]["height"]}'
      f'  {flux_baseline["primary"]["bytes"]} bytes  (nao modificada)')
print()
print('REFERENCES:')
print('    full_body.png')
print('    outfit.png')
print('    (face.png fora: o node Core comporta 3 imagens no total)')
print()
print('MODEL:')
print('    Qwen-Image-Edit-2511  (fp8mixed)')
print('    revision 6f3ccc0b56e431dc6a0c2b2039706d7d26f22cb9')
print()
print('DENOISE:')
print(f'    {DENOISE}   <- baseline, {DENOISE_STATUS}')
print('    nao calibrado nesta rodada, por instrucao')
print()
print('SEED:', SEED)
print('=' * 68)
print('objetivo: manter identidade do run_003, preservar design original,')
print('aproximar proporcoes do chibi, sem inventar roupa nem perder')
print('cabelo, chifres e ornamentos.')
print('=' * 68)


## 8 — PRIMARY EXPERIMENT · FLUX run_003 → Qwen

**Este é o experimento principal.**

```
Original → FLUX run_003 (já existe) → Qwen → resultado final
```

`image1` = saída do `run_003`; `image2` = `full_body`; `image3` = `outfit`.
`primary_image_role` deve sair como `stage1_output` — **não** `full_body`.


In [ ]:
%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model qwen-refiner --character waifu_001 \
    --input "{FLUX_RUN003}" \
    --ref reference/full_body.png --ref reference/outfit.png \
    --seed {SEED} --denoise {DENOISE} \
    --prompt "{PROMPT_QWEN}"


## 9 — Comparação A · Original → Qwen

Isola o Qwen **sozinho**, sem o FLUX: mostra o quanto do resultado principal
vem do estágio 1. `primary_image_role` deve sair como `full_body`.


In [ ]:
%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model qwen-refiner --character waifu_001 \
    --input reference/full_body.png \
    --seed {SEED} --denoise {DENOISE} \
    --prompt "{PROMPT_QWEN}"

## 10 — Comparação B · FLUX run_003 → Qwen, uma referência

Igual ao PRIMARY, porém **sem** `outfit.png`. Isola o efeito da segunda
referência de design.


In [ ]:
%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model qwen-refiner --character waifu_001 \
    --input "{FLUX_RUN003}" \
    --ref reference/full_body.png \
    --seed {SEED} --denoise {DENOISE} \
    --prompt "{PROMPT_QWEN}"


## 11 — Comparação C · FLUX run_001 → Qwen

Mesmas referências do PRIMARY, trocando apenas a entrada do estágio 1
(`run_001` em vez de `run_003`). Isola exatamente a decisão desta correção:
partir de um chibi mais forte com design reinterpretado, ou de um design
fiel menos chibi.

Pulado automaticamente se o `run_001` não estiver no ZIP.


In [ ]:
%cd /content/ChibiCreate
if FLUX_RUN001 is None:
    print('PULADO: run_001 nao veio no ZIP. Comparacao C nao executada.')
else:
    !python -m scripts.chibi.cli experiment model-eval \
        --model qwen-refiner --character waifu_001 \
        --input "{FLUX_RUN001}" \
        --ref reference/full_body.png --ref reference/outfit.png \
        --seed {SEED} --denoise {DENOISE} \
        --prompt "{PROMPT_QWEN}"


## 12 — Organizar em `flux_to_qwen/`

Os runs saem em `model_eval/qwen_image_edit_2511/` na ordem de execução. Aqui
recebem nomes que dizem o que são, sem tocar nos experimentos do FLUX.

| pasta | o que é |
|---|---|
| `primary_run003_2refs` | **PRIMARY EXPERIMENT** |
| `cmp_a_qwen_only` | A · Qwen sozinho |
| `cmp_b_run003_1ref` | B · sem `outfit` |
| `cmp_c_run001_2refs` | C · entrada `run_001` |


In [ ]:
import pathlib, shutil, json

src = pathlib.Path('experiments/model_eval/qwen_image_edit_2511')
dst = pathlib.Path('experiments/model_eval/flux_to_qwen')
dst.mkdir(parents=True, exist_ok=True)

# Ordem de execucao das celulas acima. C so existe se run_001 veio no ZIP.
plano = [
    ('primary_run003_2refs', 'PRIMARY: flux run_003 -> qwen + full_body + outfit',
     'run_003', True),
    ('cmp_a_qwen_only', 'A: original -> qwen (sem flux)', None, False),
    ('cmp_b_run003_1ref', 'B: flux run_003 -> qwen + full_body', 'run_003', True),
]
if FLUX_RUN001 is not None:
    plano.append(('cmp_c_run001_2refs',
                  'C: flux run_001 -> qwen + full_body + outfit', 'run_001', True))

runs = sorted(src.glob('run_*'))
assert len(runs) >= len(plano), (
    f'esperava {len(plano)} runs, achei {len(runs)}: {[r.name for r in runs]}')

for (nome, desc, estagio1, usa_flux), r in zip(plano, runs[-len(plano):]):
    alvo = dst / nome
    if alvo.exists():
        shutil.rmtree(alvo)
    shutil.copytree(r, alvo)
    rec = json.load(open(alvo / 'recipe.json'))
    rec['experiment'] = 'flux_to_qwen'
    rec['test_label'] = desc
    rec['is_primary_experiment'] = nome.startswith('primary')
    rec['denoise_status'] = DENOISE_STATUS
    rec['stage1_source_run'] = estagio1
    rec['stage1_baseline'] = (
        flux_baseline['primary'] if estagio1 == 'run_003'
        else flux_baseline.get('run_001') if estagio1 == 'run_001'
        else None)
    json.dump(rec, open(alvo / 'recipe.json', 'w'), indent=2, ensure_ascii=False)

    esperado = 'stage1_output' if usa_flux else 'full_body'
    obtido = rec.get('primary_image_role')
    marca = 'PRIMARY  ' if rec['is_primary_experiment'] else '         '
    print(f'{marca}{nome}')
    print(f'   {desc}')
    print(f'   primary_image_role : {obtido}'
          f'{"" if obtido == esperado else f"  <-- ESPERAVA {esperado}"}')
    print(f'   reference_count    : {rec.get("reference_count")}')


## 13 — Comparação visual e tabela

O PRIMARY aparece primeiro, logo abaixo do original e da entrada `run_003`.


In [ ]:
import pathlib, json
from PIL import Image
from IPython.display import display

orig = pathlib.Path('characters/waifu_001/reference/full_body.png')
base = pathlib.Path('experiments/model_eval/flux_to_qwen')

print('=' * 70); print('ORIGINAL (arte-fonte)')
display(Image.open(orig).resize((256, 256)))
print('=' * 70); print('ENTRADA DO ESTAGIO 2 — FLUX run_003')
display(Image.open(FLUX_RUN003).resize((256, 256)))

ordem = ['primary_run003_2refs', 'cmp_a_qwen_only',
         'cmp_b_run003_1ref', 'cmp_c_run001_2refs']
for nome in ordem:
    r = base / nome
    if not r.is_dir():
        continue
    rec = json.load(open(r / 'recipe.json'))
    print('=' * 70)
    tag = '*** PRIMARY EXPERIMENT ***  ' if rec.get('is_primary_experiment') else ''
    print(f'{tag}{nome}')
    print(f'   {rec.get("test_label")}')
    for k in ('primary_image_role', 'stage1_source_run', 'reference_count',
              'seed', 'execution_time', 'output_sha256', 'output_pixel_sha256'):
        v = rec.get(k)
        if isinstance(v, str) and len(v) > 24:
            v = v[:24] + '...'
        print(f'   {k:22} {v}')
    print(f'   parameters             {rec.get("parameters")}')
    out = r / 'output.png'
    if out.is_file():
        display(Image.open(out).resize((256, 256)))


## 14 — Avaliação: os três eixos

Olhe o **PRIMARY** primeiro; as comparações servem para explicar o que ele
produziu. Preencha à mão — não há fórmula, e OVERALL não é média aritmética.

Perguntas desta rodada, na ordem declarada do objetivo:

1. **IDENTITY** — continua a mesma personagem do `run_003`?
2. **DESIGN_PRESERVATION** — roupa, capa, chifres, cabelo e ornamentos
   sobreviveram, nas mesmas formas e posições?
3. **STYLE** — as proporções chegaram mais perto do chibi alvo?
4. Alguma peça de roupa foi **inventada** ou substituída?
5. O que a comparação C mostra: partir do `run_003` foi mesmo melhor que
   partir do `run_001`?

*Simplificar é remover detalhe. Redesenhar é trocar o design.* Um chibi mais
limpo não é perda de design; uma capa que virou outra capa é.

**A decisão artística é humana.** O agente não escolhe vencedor, não aprova
Chibi Master e não julga beleza.


## 15 — Baixar os resultados


In [ ]:
import shutil, json, pathlib

meta = {
    'experiment': 'flux_to_qwen',
    'primary_experiment': 'flux run_003 -> qwen + full_body + outfit',
    'primary_experiment_rationale': (
        'run_003 preserva melhor design/roupa/capa/cabelo/ornamentos que '
        'run_001, embora esteja menos chibi. O estagio Qwen existe para '
        'aproximar as proporcoes do chibi sem perder esse design.'),
    'infrastructure': 'google_colab',
    'infrastructure_status': 'EXPERIMENTAL_TEMPORARY',
    'gpu': json.load(open('/content/gpu_info.json')),
    'models': json.load(open('/content/model_record.json')),
    'comfyui_commit': COMFY_COMMIT,
    'stage1_baseline': json.load(open('/content/flux_baseline.json')),
    'denoise': DENOISE,
    'denoise_status': DENOISE_STATUS,
    'seed': SEED,
    'custom_nodes': [],
    'custom_nodes_note': 'Nenhum. TextEncodeQwenImageEditPlus e Core (3 imagens).',
    'cost': 0,
    'cost_note': 'Nenhum custo direto observado nesta sessao.',
    'cost_warning': 'Nao extrapolar para custo de producao.',
    'limitations': [
        'denoise NAO calibrado (BASELINE_HYPOTHESIS)',
        'seed nao atravessa estagios: 42 no Qwen nao reproduz o ruido do FLUX',
        'face.png fora de todas as rodadas: node Core comporta 3 imagens',
        'escolha de run_003 vem de inspecao visual humana, nao de metrica',
        'comparacao C pulada se run_001 nao vier no ZIP',
        'um run por teste: nao afirma determinismo',
        'VRAM via /system_stats e aproximacao, nao pico',
    ],
    'approval_status': 'experimental',
}
d = pathlib.Path('experiments/model_eval/flux_to_qwen')
json.dump(meta, open(d / 'session_metadata.json', 'w'), indent=2, ensure_ascii=False)

shutil.make_archive('/content/flux_to_qwen_results', 'zip', d)
print('zip:', pathlib.Path('/content/flux_to_qwen_results.zip').stat().st_size / 1e6, 'MB')

from google.colab import files
files.download('/content/flux_to_qwen_results.zip')

---

## PARE AQUI

Três execuções — fim do escopo.

Não: LoRA · ControlNet · animação · `master.png` · Flow 02 · Step1X · WAI ·
calibração de denoise · candidatos extras.

### [HUMAN REVIEW REQUIRED]

A avaliação artística é humana. Me mande o ZIP (ou commite
`experiments/model_eval/flux_to_qwen/`) que eu faço a comparação nos três
eixos e o relatório.